In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import urllib.request
import io

In [37]:
url = "https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip"
response = urllib.request.urlopen(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.read()))
data = pd.read_csv(zip_file.open("energydata_complete.csv"))
print("Dataset downloaded successfully")

Dataset downloaded successfully


In [38]:
print(data.head())
print(data.shape)
print(data.columns)
print(data.info())

                  date  Appliances  lights     T1       RH_1    T2       RH_2  \
0  2016-01-11 17:00:00          60      30  19.89  47.596667  19.2  44.790000   
1  2016-01-11 17:10:00          60      30  19.89  46.693333  19.2  44.722500   
2  2016-01-11 17:20:00          50      30  19.89  46.300000  19.2  44.626667   
3  2016-01-11 17:30:00          50      40  19.89  46.066667  19.2  44.590000   
4  2016-01-11 17:40:00          60      40  19.89  46.333333  19.2  44.530000   

      T3       RH_3         T4  ...         T9   RH_9     T_out  Press_mm_hg  \
0  19.79  44.730000  19.000000  ...  17.033333  45.53  6.600000        733.5   
1  19.79  44.790000  19.000000  ...  17.066667  45.56  6.483333        733.6   
2  19.79  44.933333  18.926667  ...  17.000000  45.50  6.366667        733.7   
3  19.79  45.000000  18.890000  ...  17.000000  45.40  6.250000        733.8   
4  19.79  45.000000  18.890000  ...  17.000000  45.40  6.133333        733.9   

   RH_out  Windspeed  Visibility

In [39]:
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values("date")
data = data.set_index("date")
print(data.head())

                     Appliances  lights     T1       RH_1    T2       RH_2  \
date                                                                         
2016-01-11 17:00:00          60      30  19.89  47.596667  19.2  44.790000   
2016-01-11 17:10:00          60      30  19.89  46.693333  19.2  44.722500   
2016-01-11 17:20:00          50      30  19.89  46.300000  19.2  44.626667   
2016-01-11 17:30:00          50      40  19.89  46.066667  19.2  44.590000   
2016-01-11 17:40:00          60      40  19.89  46.333333  19.2  44.530000   

                        T3       RH_3         T4       RH_4  ...         T9  \
date                                                         ...              
2016-01-11 17:00:00  19.79  44.730000  19.000000  45.566667  ...  17.033333   
2016-01-11 17:10:00  19.79  44.790000  19.000000  45.992500  ...  17.066667   
2016-01-11 17:20:00  19.79  44.933333  18.926667  45.890000  ...  17.000000   
2016-01-11 17:30:00  19.79  45.000000  18.890000  45.72333

In [40]:
print("Missing Values")
print(data.isnull().sum())
print("Duplicate Values")
print(data.index.duplicated().sum())
data = data[~data.index.duplicated(keep="first")]

Missing Values
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64
Duplicate Values
0


In [41]:
time_difference = data.index.to_series().diff()
print(time_difference.value_counts().head())
full_time = pd.date_range(
    start=data.index.min(),
    end=data.index.max(),
    freq="10min"
)
data = data.reindex(full_time)
data.index.name = "date"
print("Total Records:", len(data))
print("Missing Records:", data.isnull().sum().sum())

date
0 days 00:10:00    19734
Name: count, dtype: int64
Total Records: 19735
Missing Records: 0


In [42]:
data = data.interpolate(method="time")
data = data.ffill()
data = data.bfill()
data = data.dropna()
print(data.isnull().sum().sum())
print("Data Cleaning Completed")

0
Data Cleaning Completed


In [43]:
hourly_data = data.resample("h").agg({
    "Appliances": "sum",
    "lights": "sum",
    "T_out": "mean",
    "RH_out": "mean",
    "Windspeed": "mean"
})
hourly_data["Energy_kWh"] = hourly_data["Appliances"] / 1000
daily_data = hourly_data["Energy_kWh"].resample("D").sum()
print(hourly_data.head())
print(daily_data.head())

                     Appliances  lights     T_out     RH_out  Windspeed  \
date                                                                      
2016-01-11 17:00:00         330     210  6.308333  92.000000   6.166667   
2016-01-11 18:00:00        1060     310  5.941667  91.583333   5.416667   
2016-01-11 19:00:00        1040     150  6.000000  89.750000   6.000000   
2016-01-11 20:00:00         750     210  6.000000  87.583333   6.000000   
2016-01-11 21:00:00         620     140  5.833333  87.416667   6.000000   

                     Energy_kWh  
date                             
2016-01-11 17:00:00        0.33  
2016-01-11 18:00:00        1.06  
2016-01-11 19:00:00        1.04  
2016-01-11 20:00:00        0.75  
2016-01-11 21:00:00        0.62  
date
2016-01-11     5.74
2016-01-12    12.34
2016-01-13    13.97
2016-01-14    21.80
2016-01-15    18.05
Freq: D, Name: Energy_kWh, dtype: float64


In [44]:
peak_load = hourly_data["Energy_kWh"].max()
average_load = hourly_data["Energy_kWh"].mean()
print("Peak Load:", peak_load, "kW")
print("Average Load:", average_load, "kW")
print("90% Quantile:",
      np.percentile(hourly_data["Energy_kWh"], 90))
print("95% Quantile:",
      np.percentile(hourly_data["Energy_kWh"], 95))
print("99% Quantile:",
      np.percentile(hourly_data["Energy_kWh"], 99))

Peak Load: 3.65 kW
Average Load: 0.5860212765957447 kW
90% Quantile: 1.23
95% Quantile: 1.66
99% Quantile: 2.5211000000000015


In [45]:
average = hourly_data["Energy_kWh"].mean()
std = hourly_data["Energy_kWh"].std()
limit = average + 3 * std
anomalies = hourly_data[
    hourly_data["Energy_kWh"] > limit
]
hourly_data["Rolling_Average"] = (
    hourly_data["Energy_kWh"].rolling(24).mean()
)
print("Total Anomalies:", len(anomalies))
print(anomalies.head())

Total Anomalies: 89
                     Appliances  lights     T_out     RH_out  Windspeed  \
date                                                                      
2016-01-14 10:00:00        3230      80  2.066667  94.166667   5.166667   
2016-01-14 17:00:00        2430      30  4.558333  76.000000   6.583333   
2016-01-16 18:00:00        2840     140  2.291667  81.750000   5.333333   
2016-01-16 19:00:00        2350     110  1.500000  89.083333   2.166667   
2016-01-17 07:00:00        2090      40 -0.041667  96.000000   2.583333   

                     Energy_kWh  
date                             
2016-01-14 10:00:00        3.23  
2016-01-14 17:00:00        2.43  
2016-01-16 18:00:00        2.84  
2016-01-16 19:00:00        2.35  
2016-01-17 07:00:00        2.09  


In [46]:
hourly_data["Day"] = hourly_data.index.dayofweek
hourly_data["Day_Type"] = np.where(
    hourly_data["Day"] >= 5,
    "Weekend",
    "Weekday"
)
comparison = hourly_data.groupby(
    "Day_Type"
)["Energy_kWh"].mean()
hourly_data["Month"] = hourly_data.index.month
monthly = hourly_data.groupby(
    "Month"
)["Energy_kWh"].sum()
hourly_data["Season"] = hourly_data.index.month.map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})
seasonal = hourly_data.groupby(
    "Season"
)["Energy_kWh"].mean()
print(comparison)
print(monthly)
print(seasonal)

Day_Type
Weekday    0.579323
Weekend    0.603487
Name: Energy_kWh, dtype: float64
Month
1    283.51
2    421.55
3    432.80
4    427.20
5    362.95
Name: Energy_kWh, dtype: float64
Season
Spring    0.580422
Winter    0.595993
Name: Energy_kWh, dtype: float64


In [47]:
correlation = hourly_data[[
    "Energy_kWh",
    "T_out",
    "RH_out",
    "Windspeed"
]].corr()
print("Weather Relationship:")
print(correlation)
load_factor = average_load / peak_load
print("Load Factor:", load_factor)
print("Load Factor Percentage:", load_factor * 100)

Weather Relationship:
            Energy_kWh     T_out    RH_out  Windspeed
Energy_kWh    1.000000  0.124036 -0.192946   0.112784
T_out         0.124036  1.000000 -0.574353   0.193064
RH_out       -0.192946 -0.574353  1.000000  -0.175334
Windspeed     0.112784  0.193064 -0.175334   1.000000
Load Factor: 0.16055377440979307
Load Factor Percentage: 16.055377440979306


In [48]:
print("Peak Load:", peak_load)
print("Average Load:", average_load)
print("Load Factor:", load_factor * 100, "%")
print("Total Anomalies:", len(anomalies))
print("Weekday and Weekend Comparison:")
print(comparison)
print("Monthly Consumption:")
print(monthly)
print("Seasonal Consumption:")
print(seasonal)
print("Weather Correlation:")
print(correlation)
print("Energy Saved:", saved_energy, "kWh")
data.to_csv("cleaned_energy_data.csv")
hourly_data.to_csv("hourly_energy_data.csv")
anomalies.to_csv("anomaly_log.csv")
print("Project Completed Successfully!")

Peak Load: 3.65
Average Load: 0.5860212765957447
Load Factor: 16.055377440979306 %
Total Anomalies: 89
Weekday and Weekend Comparison:
Day_Type
Weekday    0.579323
Weekend    0.603487
Name: Energy_kWh, dtype: float64
Monthly Consumption:
Month
1    283.51
2    421.55
3    432.80
4    427.20
5    362.95
Name: Energy_kWh, dtype: float64
Seasonal Consumption:
Season
Spring    0.580422
Winter    0.595993
Name: Energy_kWh, dtype: float64
Weather Correlation:
            Energy_kWh     T_out    RH_out  Windspeed
Energy_kWh    1.000000  0.124036 -0.192946   0.112784
T_out         0.124036  1.000000 -0.574353   0.193064
RH_out       -0.192946 -0.574353  1.000000  -0.175334
Windspeed     0.112784  0.193064 -0.175334   1.000000
Energy Saved: 59.603 kWh
Project Completed Successfully!
